# Minimal LoRA Dual-Head XLM-R with Consistency

Train one LoRA `xlm-roberta-base` backbone with two heads: a 5-class classifier `p(k | x)` and a scalar regressor `s(x)`. The loss is

$$L = L_{CE} + \lambda L_{Huber} + \gamma\left(s(x) - \sum_k k p(k | x)\right)^2.$$

The consistency term makes the two heads agree in expectation while still letting each head learn a useful ordinal view of the label.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import json
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, Sequence, Value
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoTokenizer, Trainer, TrainingArguments, XLMRobertaModel, XLMRobertaPreTrainedModel, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_dual_head_consistency_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

EPOCHS = 1
BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 1
EVAL_BATCH_SIZE = 1024
LR = 1.5e-4
FP16 = torch.cuda.is_available()

# Loss weights: CE + LAMBDA_HUBER * Huber + GAMMA_CONSISTENCY * agreement.
LAMBDA_HUBER = 0.50
GAMMA_CONSISTENCY = 0.20
HUBER_DELTA = 1.0

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data and tokenization

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")
df["lang"] = df.get("lang", "unk")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
y_val = val_df["label"].to_numpy(dtype=int)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def one_hot(label, n_classes=N_CLASSES):
    vec = [0.0] * n_classes
    vec[int(label)] = 1.0
    return vec


def tokenize_dual(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["class_labels"] = [one_hot(x) for x in batch["label"]]
    out["score_labels"] = [float(x) for x in batch["label"]]
    langs = batch["lang"] if "lang" in batch else ["unk"] * len(batch["sentence"])
    out["lang"] = [0 if x == "eng_Latn" else 1 for x in langs]
    return out


def to_hf_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize_dual, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("class_labels", Sequence(Value("float32"), length=N_CLASSES))
    ds = ds.cast_column("score_labels", Value("float32"))
    ds = ds.cast_column("lang", Value("int64"))
    ds.set_format("torch")
    return ds


train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)

## Model and trainer

In [ ]:
class XLMRDualHeadConsistencyModel(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = XLMRobertaModel(config)
        hidden = config.hidden_size
        self.num_labels = config.num_labels
        self.lambda_huber = getattr(config, "lambda_huber", LAMBDA_HUBER)
        self.gamma_consistency = getattr(config, "gamma_consistency", GAMMA_CONSISTENCY)
        self.huber_delta = getattr(config, "huber_delta", HUBER_DELTA)
        self.register_buffer("class_values", torch.arange(self.num_labels, dtype=torch.float32), persistent=False)

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, self.num_labels),
        )
        self.regressor = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, 1),
        )
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, class_labels=None, score_labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0, :])
        class_logits = self.classifier(pooled)
        scores = self.regressor(pooled).squeeze(-1)
        probs = F.softmax(class_logits, dim=-1)
        expected_scores = probs @ self.class_values.to(probs.device)

        loss = None
        if class_labels is not None and score_labels is not None:
            class_labels = class_labels.float()
            score_labels = score_labels.float().view_as(scores)
            log_probs = F.log_softmax(class_logits, dim=-1)
            ce_loss = -(class_labels * log_probs).sum(dim=-1).mean()
            huber_loss = F.huber_loss(scores, score_labels, delta=self.huber_delta)
            consistency_loss = torch.square(scores - expected_scores).mean()
            loss = ce_loss + self.lambda_huber * huber_loss + self.gamma_consistency * consistency_loss

        # Trainer expects one prediction tensor. Columns 0..4 are classifier logits; column 5 is s(x).
        logits = torch.cat([class_logits, scores.unsqueeze(-1)], dim=-1)
        return {"loss": loss, "logits": logits}


def make_model():
    config = AutoConfig.from_pretrained(MODEL_ID, num_labels=N_CLASSES)
    config.lambda_huber = LAMBDA_HUBER
    config.gamma_consistency = GAMMA_CONSISTENCY
    config.huber_delta = HUBER_DELTA
    model = XLMRDualHeadConsistencyModel.from_pretrained(MODEL_ID, config=config)
    lora_config = LoraConfig(
        r=128,
        lora_alpha=64,
        target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
        modules_to_save=["classifier", "regressor"],
        lora_dropout=0.01,
        task_type="SEQ_CLS",
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(cls - classes), axis=1) for cls in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)


def apply_thresholds(scores, thresholds):
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    thresholds = np.asarray(thresholds, dtype=np.float32)
    return np.searchsorted(thresholds, scores, side="right").astype(int)


def tune_mae_thresholds(scores, labels, n_classes=N_CLASSES):
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    unique_scores, group_starts = np.unique(sorted_scores, return_index=True)
    group_ends = np.r_[group_starts[1:], len(sorted_scores)]
    n_groups = len(unique_scores)

    group_cost = np.zeros((n_classes, n_groups), dtype=np.float64)
    for g, (start, end) in enumerate(zip(group_starts, group_ends)):
        y = sorted_labels[start:end]
        for cls in range(n_classes):
            group_cost[cls, g] = np.abs(cls - y).sum()

    prefix_cost = np.c_[np.zeros(n_classes), np.cumsum(group_cost, axis=1)]
    dp = np.full((n_classes, n_groups + 1), np.inf, dtype=np.float64)
    back = np.zeros((n_classes, n_groups + 1), dtype=int)
    dp[0] = prefix_cost[0]

    for cls in range(1, n_classes):
        best_value = np.inf
        best_split = 0
        for j in range(n_groups + 1):
            candidate = dp[cls - 1, j] - prefix_cost[cls, j]
            if candidate < best_value:
                best_value = candidate
                best_split = j
            dp[cls, j] = prefix_cost[cls, j] + best_value
            back[cls, j] = best_split

    cuts = []
    j = n_groups
    for cls in range(n_classes - 1, 0, -1):
        j = back[cls, j]
        cuts.append(j)
    cuts = cuts[::-1]

    thresholds = []
    eps = 1e-6
    for cut in cuts:
        if cut <= 0:
            thresholds.append(float(unique_scores[0] - eps))
        elif cut >= n_groups:
            thresholds.append(float(unique_scores[-1] + eps))
        else:
            thresholds.append(float((unique_scores[cut - 1] + unique_scores[cut]) / 2.0))
    return np.array(thresholds, dtype=np.float32), apply_thresholds(scores, thresholds)


def split_predictions(predictions):
    predictions = np.asarray(predictions)
    class_logits = predictions[:, :N_CLASSES]
    scores = predictions[:, N_CLASSES]
    probs = softmax_np(class_logits)
    expected_scores = probs @ np.arange(N_CLASSES)
    return class_logits, probs, scores, expected_scores


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    class_labels, score_labels = labels
    y_true = np.asarray(score_labels).reshape(-1)
    _, probs, scores, expected_scores = split_predictions(predictions)
    map_preds = probs.argmax(axis=1).astype(int)
    bayes_preds = bayes_mae_decode(probs)
    reg_preds = np.rint(np.clip(scores, 0, N_CLASSES - 1)).astype(int)
    avg_scores = (np.clip(scores, 0, N_CLASSES - 1) + expected_scores) / 2.0
    avg_preds = np.rint(np.clip(avg_scores, 0, N_CLASSES - 1)).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, map_preds)),
        "map_mae": float(mean_absolute_error(y_true, map_preds)),
        "bayes_mae": float(mean_absolute_error(y_true, bayes_preds)),
        "reg_rounded_mae": float(mean_absolute_error(y_true, reg_preds)),
        "expected_score_mae": float(mean_absolute_error(y_true, expected_scores)),
        "reg_score_mae": float(mean_absolute_error(y_true, scores)),
        "avg_rounded_mae": float(mean_absolute_error(y_true, avg_preds)),
        "head_agreement_mse": float(np.mean(np.square(scores - expected_scores))),
    }


def make_training_args(run_name):
    kwargs = dict(
        output_dir=str(OUTPUT_DIR / run_name / "checkpoints"),
        overwrite_output_dir=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=EPOCHS,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        label_names=["class_labels", "score_labels"],
        seed=SEED,
    )
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)

## Train

In [ ]:
def debug_one_batch(model, dataset):
    batch = dataset.select(range(min(2, len(dataset))))[:]
    device = next(model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items() if hasattr(v, "to")}
    model.train()
    model.zero_grad(set_to_none=True)
    out = model(**batch)
    loss = out["loss"]
    print("preflight loss:", float(loss.detach().cpu()))
    loss.backward()

    grad_sums = {"classifier": 0.0, "regressor": 0.0, "lora": 0.0}
    for name, param in model.named_parameters():
        if param.grad is None:
            continue
        grad = float(param.grad.detach().abs().sum().cpu())
        if "classifier" in name:
            grad_sums["classifier"] += grad
        if "regressor" in name:
            grad_sums["regressor"] += grad
        if "lora_" in name:
            grad_sums["lora"] += grad
    model.zero_grad(set_to_none=True)
    print("preflight grad sums:", grad_sums)
    return grad_sums


model = make_model()
if torch.cuda.is_available():
    model.to("cuda")
debug_one_batch(model, train_ds)
trainer = Trainer(
    model=model,
    args=make_training_args("dual_head_consistency_1epoch"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
# metrics = trainer.evaluate()
# metrics

## Validation decoding diagnostics

In [ ]:
val_predictions = trainer.predict(val_ds).predictions
val_class_logits, val_probs, val_scores, val_expected = split_predictions(val_predictions)
val_scores_clipped = np.clip(val_scores, 0, N_CLASSES - 1)
val_avg_scores = (val_scores_clipped + val_expected) / 2.0

map_labels = val_probs.argmax(axis=1).astype(int)
bayes_labels = bayes_mae_decode(val_probs)
reg_round_labels = np.rint(val_scores_clipped).astype(int)
avg_round_labels = np.rint(np.clip(val_avg_scores, 0, N_CLASSES - 1)).astype(int)
reg_thresholds, reg_tuned_labels = tune_mae_thresholds(val_scores_clipped, y_val)
avg_thresholds, avg_tuned_labels = tune_mae_thresholds(val_avg_scores, y_val)

summary = pd.DataFrame(
    [
        {"decoder": "classifier_map", "mae": mean_absolute_error(y_val, map_labels), "counts": np.bincount(map_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "classifier_bayes_mae", "mae": mean_absolute_error(y_val, bayes_labels), "counts": np.bincount(bayes_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "regressor_round", "mae": mean_absolute_error(y_val, reg_round_labels), "counts": np.bincount(reg_round_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "regressor_tuned_thresholds", "mae": mean_absolute_error(y_val, reg_tuned_labels), "counts": np.bincount(reg_tuned_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "average_score_round", "mae": mean_absolute_error(y_val, avg_round_labels), "counts": np.bincount(avg_round_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "average_score_tuned_thresholds", "mae": mean_absolute_error(y_val, avg_tuned_labels), "counts": np.bincount(avg_tuned_labels, minlength=N_CLASSES).tolist()},
    ]
)
display(summary.sort_values("mae"))

print("reg thresholds:", reg_thresholds.tolist())
print("avg thresholds:", avg_thresholds.tolist())
print("head agreement MSE:", float(np.mean(np.square(val_scores - val_expected))))
print("corr s(x), E[y]:", float(np.corrcoef(val_scores, val_expected)[0, 1]))

In [ ]:
probe = pd.DataFrame(
    {
        "y": y_val,
        "score_s": val_scores,
        "expected_from_classifier": val_expected,
        "avg_score": val_avg_scores,
        "classifier_bayes": bayes_labels,
        "reg_tuned": reg_tuned_labels,
        "avg_tuned": avg_tuned_labels,
        "head_disagreement": val_scores - val_expected,
        "bayes_err": np.abs(bayes_labels - y_val),
        "reg_tuned_err": np.abs(reg_tuned_labels - y_val),
        "avg_tuned_err": np.abs(avg_tuned_labels - y_val),
    }
)

display(
    probe.assign(abs_head_disagreement=probe["head_disagreement"].abs())
    .sort_values("abs_head_disagreement", ascending=False)
    .head(20)
)

pd.crosstab(
    pd.Series(bayes_labels, name="classifier_bayes"),
    pd.Series(avg_tuned_labels, name="avg_tuned"),
    margins=True,
)

## Save model and consistency config

In [ ]:
final_dir = OUTPUT_DIR / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config_path = OUTPUT_DIR / "dual_head_consistency_config.json"
config_path.write_text(
    json.dumps(
        {
            "model_id": MODEL_ID,
            "n_classes": N_CLASSES,
            "lambda_huber": LAMBDA_HUBER,
            "gamma_consistency": GAMMA_CONSISTENCY,
            "huber_delta": HUBER_DELTA,
            "reg_thresholds": reg_thresholds.tolist() if "reg_thresholds" in globals() else None,
            "avg_thresholds": avg_thresholds.tolist() if "avg_thresholds" in globals() else None,
            "recommended_decoder": "average_score_tuned_thresholds",
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("model:", final_dir)
print("config:", config_path)

## Optional submission

This writes two useful submission candidates: classifier Bayes-MAE decoding and the tuned average of `s(x)` with `E[y]`.

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    test_predictions = trainer.predict(test_ds).predictions
    _, test_probs, test_scores, test_expected = split_predictions(test_predictions)
    test_scores_clipped = np.clip(test_scores, 0, N_CLASSES - 1)
    test_avg_scores = (test_scores_clipped + test_expected) / 2.0

    test_bayes = bayes_mae_decode(test_probs)
    test_avg_tuned = apply_thresholds(test_avg_scores, avg_thresholds if "avg_thresholds" in globals() else np.arange(0.5, N_CLASSES - 1, 1.0))

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    bayes_path = OUTPUT_DIR / "submission_dual_head_bayes_mae.csv"
    avg_path = OUTPUT_DIR / "submission_dual_head_avg_tuned.csv"
    pd.DataFrame({"id": test_df["id"], "label": test_bayes.astype(int)}).to_csv(bayes_path, index=False)
    pd.DataFrame({"id": test_df["id"], "label": test_avg_tuned.astype(int)}).to_csv(avg_path, index=False)
    print("bayes counts:", np.bincount(test_bayes, minlength=N_CLASSES).tolist())
    print("avg tuned counts:", np.bincount(test_avg_tuned, minlength=N_CLASSES).tolist())
    print(bayes_path)
    print(avg_path)
else:
    print("No test CSV found:", TEST_CSV)